In [1]:
import pandas as pd
df = pd.read_csv("../data/framingham.csv")
df.shape

(4238, 16)

In [2]:
df.head()

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


In [3]:
df.isna().sum()

male                 0
age                  0
education          105
currentSmoker        0
cigsPerDay          29
BPMeds              53
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol             50
sysBP                0
diaBP                0
BMI                 19
heartRate            1
glucose            388
TenYearCHD           0
dtype: int64

In [4]:
df["TenYearCHD"].value_counts(normalize=True)

TenYearCHD
0    0.848042
1    0.151958
Name: proportion, dtype: float64

In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
male,4238.0,0.429212,0.495022,0.00,0.00,0.0,1.000,1.0
age,4238.0,49.584946,8.572160,32.00,42.00,49.0,56.000,70.0
education,4133.0,1.978950,1.019791,1.00,1.00,2.0,3.000,4.0
currentSmoker,4238.0,0.494101,0.500024,0.00,0.00,0.0,1.000,1.0
cigsPerDay,4209.0,9.003089,11.920094,0.00,0.00,0.0,20.000,70.0
BPMeds,4185.0,0.029630,0.169584,0.00,0.00,0.0,0.000,1.0
prevalentStroke,4238.0,0.005899,0.076587,0.00,0.00,0.0,0.000,1.0
prevalentHyp,4238.0,0.310524,0.462763,0.00,0.00,0.0,1.000,1.0
diabetes,4238.0,0.025720,0.158316,0.00,0.00,0.0,0.000,1.0
totChol,4188.0,236.721585,44.590334,107.00,206.00,234.0,263.000,696.0


## Step 2 – Clean

In [6]:
# Non-smokers with missing cigarette count -> 0
df.loc[(df["currentSmoker"] == 0) & (df["cigsPerDay"].isna()), "cigsPerDay"] = 0

# Education is not a clinical risk factor
df = df.drop(columns=["education"])

df.isna().sum()

male                 0
age                  0
currentSmoker        0
cigsPerDay          29
BPMeds              53
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol             50
sysBP                0
diaBP                0
BMI                 19
heartRate            1
glucose            388
TenYearCHD           0
dtype: int64

## Step 3 – Split

In [7]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["TenYearCHD"])
y = df["TenYearCHD"]

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42)
X_cal, X_test, y_cal, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

print("Train:", X_train.shape, " Cal:", X_cal.shape, " Test:", X_test.shape)
print("Positive rate:", round(y_train.mean(), 3), round(y_cal.mean(), 3), round(y_test.mean(), 3))

Train: (2542, 14)  Cal: (848, 14)  Test: (848, 14)
Positive rate: 0.152 0.152 0.152


## Step 4 – Logistic Regression (baseline)

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

lr = Pipeline([
    ("impute", SimpleImputer(strategy="median")),   # fill missing values
    ("scale", StandardScaler()),                    # put all features on same scale
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
])

lr.fit(X_train, y_train)

p_lr_test = lr.predict_proba(X_test)[:, 1]
print("Logistic Regression AUC:", round(roc_auc_score(y_test, p_lr_test), 3))

Logistic Regression AUC: 0.696


## Step 5 – XGBoost

In [9]:
from xgboost import XGBClassifier

# Weight for the minority class (about 5.6 healthy patients per sick patient)
w = (y_train == 0).sum() / (y_train == 1).sum()
print("Class weight:", round(w, 2))

xgb = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        n_estimators=300,      # number of trees
        max_depth=3,           # how deep each tree can go
        learning_rate=0.03,    # how fast it learns
        subsample=0.8,         # use 80% of rows per tree (reduces overfitting)
        colsample_bytree=0.8,  # use 80% of features per tree
        scale_pos_weight=w,    # handle the 15% imbalance
        random_state=42,
    )),
])

xgb.fit(X_train, y_train)

p_xgb_test = xgb.predict_proba(X_test)[:, 1]
print("XGBoost AUC:", round(roc_auc_score(y_test, p_xgb_test), 3))

Class weight: 5.59
XGBoost AUC: 0.679
